## End-to-End Testing (System Testing)

This notebook covers **End-to-End Testing** (**System Testing**) with [xUnit](https://xunit.net), [FluentAssertions](https://fluentassertions.com) and [Playwright](https://playwright.dev/dotnet).

We will explore end-to-end testing (system testing) in Visual Studio Code together in this notebook, where we will test our entire `Todo` application (system) end-to-end (from the frontend to the backend).

In this notebook, we will end-to-end test the `Flixtube` application.

- First we will examine existing end-to-end tests for the `Flixtube` application.
- The writing additional end-to-end tests for the `Flixtube` application will be left as an exercise.

---

### Install the Playwright dotnet Tool

Let's make sure the Playwright dotnet Tool is installed globally on your computer:

- Open VSCode's terminal (Windows/Linux: `Ctrl + J`, Mac: `Cmd + J`), or via the main menu `Terminal -> New Terminal`.
- Execute the command below in your terminal:

  ```bash
  dotnet tool install --global Microsoft.Playwright.CLI
  ```

  You can also run the cell below to run the command via your operating system's shell.

In [1]:
!dotnet tool install --global Microsoft.Playwright.CLI

Tool 'microsoft.playwright.cli' is already installed.


---

### Install the Playwright VSCode Extension

<style>
    .container {
        width: 98%;
        margin-left: 0; /* Push the container to the left */
        margin-right: auto; 
    }
    .text-image {
        margin-bottom: 35px; /* Space between sections */
        overflow: hidden; /* Ensure image stays within the container */
    }
    .text {
        text-align: justify; /* Justify the text for better readability */
    }
    .image {
        float: right; /* Float the image to the right */
        margin-left: 25px; /* Space between image and text */
        margin-bottom: 10px; /* Space between image and text */
        max-width: 50%; /* Limit image size */
        height: auto; /* Maintain aspect ratio */
    }
</style>

<div class="container">
    <div class="text-image">
        <img class="image" src="notebook_images/playwright-extension.png">
        <div class="text">
            <p>
                Let's make sure the Playwright VSCode Extension is installed:
            </p>
            <ul>
                <li>Click the <img src="notebook_images/extensions-view-icon.png" /> icon for the Extensions View in VSCode's <b>Activity Bar</b>.</li>
                <li>Search for the <b>Playwright Tests for VSCode</b> extension in the Search Bar.</li>
                <li>Select the <b>Playwright Tests for VSCode</b> extension, and click the <b>Install</b> button (if it isn't already installed).</li>
            </ul>
        </div>
    </div>
</div>

<div class="container">
    <div class="text-image">
        <img class="image" src="notebook_images/playwright-tool.png">
        <div class="text">
            <ul>
                <li>Click the Test Explorer icon <img src="notebook_images/test-explorer-view-icon.png"/> in the <b>Activity Bar</b>.</li>
                <li>You should now see a <b>PLAYWRIGHT</b> section below the <b>TEST EXPLORER</b> in the <b>Primary Side Bar</b>.</li>
            </ul>
        </div>
    </div>
</div>

---

### End-to-End Testing (System Testing) Basics

In its simplest form, end-to-end testing is just integration testing, but using a Web frontend instead of an API backend.

- In **Integration Testing**, we want to **include all dependencies (services)** when testing the **Subject Under Test (SUT)**.
  - We want the SUT to use its dependencies (services), such as accessing a database, or calling an external REST API.
  - We **EXCLUDE the User Interface (UI)** when integration testing.
- In **End-to-End Testing**, we want to **include all dependencies (services)** when testing the **Subject Under Test (SUT)**.
  - We want the SUT to use its dependencies (services), such as accessing a database, or calling an external REST API.
  - We **INCLUDE the User Interface (UI)** when end-to-end testing testing.

So, to turn the integration tests we examined in the previous notebook into end-to-end test, we would simply access the SUT via the frontend user interface instead of via the backend API.

Since we are testing the complete system, Subject Under Test (SUT) can also refer to the System Under Test (SUT).

Let's see how we can use [Playwright](https://playwright.dev/dotnet), together with [xUnit](https://xunit.net) and [FluentAssertions](https://fluentassertions.com), to end-to-end test our complete application via the `Flixtube.Web` Blazor frontend.

---

### Examine the `Flixtube.Web` Microservice


In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Web -> Flixtube.Web`.
- Right-click the file `Flixtube.Web.csproj` and choose `Open to the side`.

The `Web` microservice is simply a Blazor project, without any additional NuGet packages added.

If you examine the folder structure for the `Web` microservice, you will find familiar classes when working with Blazor:

- The `Models` folder defines the two model classes:
  - `Video`, with properties `Id` and `Name`.
  - `ViewHistory`,with properties `Id`, `VideoId` and `VewedAt`.
- The `Services` folder defines a class `RestService` with the associated interface `IRestService`.
  - This class is used to communicate with the `Flixtube.Gateway` via its REST API.
- The `Program` class reads in environment variables, adds services to the Blazor host's service container (including the `RestService`), configures the HTTP Request/Response pipeline, defines a `/health` api endpoint (that can be used to check the health of the `Web` microservice), and starts the microservice listening on a specific port.
- The `Components` folder contains:
  - The usual `App.razor` and `Routes.razor` components, and the `_Imports.razor` file for defining *global usings*.
  - The `Layout` subfolder with the `MainLayout.razor` component and a `NavMenu.razor` component.
    - These two components determine the general layout and navigation interface for the Blazor frontend.
  - The `Pages` subfolder with the main user interface components:
    - The `Home.razor` component contains the user interface and functionality for the home page which displays a list of video metadata.
    - The `Upload.razor` component contains the user interface and functionality to upload a video file.
    - The `PlayVideo.razor` component contains the user interface and functionality to play a video.
    - The `DeleteVideo.razor` component contains the user interface and functionality to delete a video.
    - The `History.razor` component contains the user interface and functionality list video viewing history.
    - The `Error.razor` component contains the default user interface and functionality for errors.

Let's look at the `Home.razor` component.

- Right-click the file `Home.razor` and choose `Open to the side`.
- At the top of the file, we see the `@page` directive that sets the frontend route to `/` for the home page, and an `@inject` directive that dependency injects an instance of the `RestService` into the Razor component.
- Then the HTML with interspersed razor directives `@` for the user interface is defined:
  - It displays two button at the top of the page.
    - The `Upload Video` button navigates to the `Ùpload.razor` page with frontend route `/upload`.
    - The `Show Viewing History` button navigates to the `History.razor` page with frontend route `/history`.
  - It displays a table of video metadata beneath the two buttons.
    - The `Id` column displays a video's GUID.
    - The `Name` column displays the video's name.
    - The third column displays a `Play` button and a `Delete` button.
- Finally, at the bottom of the page, a `@code` directive includes the Razor component's functionality in C# code.
  - The `OnInitializedAsync()` method is run each time the page is displayed, where the dependency injected `RestService` is used to call the `Flixtube.Gateway` microservice's ` HTTP GET /api/metadata` endpoint which returns a list of video metadata (this is what is displayed in the HTML table).
  - The `VideoUrl(string id)` method is called when a video's `Play` button is pressed, which navigates to the `PlayVideo.razor` page.
  - The `GetDeleteModelId(Video video)` method is called when a video's `Delete` button is pressed, which displays the `DeleteVideo.razor` modal dialogue.

The other Razor components follow a similar structure, where an instance of the `RestService` is dependency injected into each component to provide a way for it to make calls to the `Flixtube.Gateway` microservice's REST API endpoints.

---

### End-To-End Testing the Application via `Flixtube.Web`

In VSCode's explorer:

- Expand the folder `flixtube -> Flixtube.Web -> Flixtube.Web.EndToEndTests`.
- Right-click the file `Flixtube.Web.EndToEndTests.csproj` and choose `Open to the side`.
- We see that this is an `xunit` project, to which we have added:
  - The `Microsoft.Playwright`, `FluentAssertions` and `Microsoft.Extensions.Configuration.Json` NuGet packages.
  - A `<CopyToOutputDirectory>` precompiler directive which copies our `appsettings.json` file from our project folder to the output directory (i.e. `bin/Debug/net9.0` for a debug build, and `bin/Release/net9.0` for a release build) when the project is compiled (built).
- Right-click the file `appsettings.json` and choose `Open to the side`.
  - Notice the setting `TestSettings.WebAppBaseUrl` with the value `http://localhost:80`.
  - This is the scheme (`http`), domain (`localhost`) and port (`80`) used to access the `Flixtube.Web` microservice once it is running.

In VSCode's explorer:

- Right-click the file `PlaywrightFixture.cs` and choose `Open to the side`.
- This file defines the class `PlaywrightFixture` that implements the interface `IAsyncLifetime`.
  - An instance of this class will be dependency injected into our test classes, and used as a *common fixture* for our tests.
- The class declares five public properties of type `IConfiguration`, `IPlaywright` and `IBrowser` (3 times), and a public property `BaseUrl` of type `string`.
- The method `InitializeAsync()` initializes the public properties.
  - `IConfiguration Configuration` holds configuration values that we read in from `appsettings.json` (in this case the Base Url to our microservice).
  - `string BaseUrl` holds the Base Url read in from `appsettings.json`. 
  - `IPlaywright Playwright` holds a Playwright instance (this is the main class defined in the `Microsoft.Playwright` NuGet package).
  - `IBrowser ChromiumBrowser` holds a `Playwright.Chromium` instance (this is a Playwright class from the `Microsoft.Playwright` NuGet package used to create a Chromium-based web browser (e.g. Chrome) for accessing the frontend's web pages).
  - `IBrowser FirefoxBrowser` holds a `Playwright.Firefox` instance (this is a Playwright class from the `Microsoft.Playwright` NuGet package used to create a Firefox-based web browser (e.g. Firefox) for accessing the frontend's web pages).
  - `IBrowser WebkitBrowser` holds a `Playwright.Webkit` instance (this is a Playwright class from the `Microsoft.Playwright` NuGet package used to create a Webkit-based web browser (e.g. Safari) for accessing the frontend's web pages).
- The method `DisposeAsync()` cleans up (frees) allocated resources by disposing (destroying) the `ChromiumBrowser`, `FirefoxBrowser`, `WebkitBrowser` and `Playwright` instances that were created in the `InitializeAsync()` method.

In VSCode's explorer:

- Right-click the file `FlixtubeTests.cs` and choose `Open to the side`.
- This file contains an `xUnit` test class with test methods.
  - It has exactly the same structure as a test class in an integration test, except that the private property of type `IAPIRequestContext` used in an intagration test has been replaced with three private properties of type `IBrowser` in the end-to-end test (`IAPIRequestContext` is used for making HTTP calls, `IBrowser` is used to interact with HTML elements on a web page).
  - It adorns the test methods with the `[Fact]` and/or `[Theory]` attributes (just as with integration tests).
  - It uses the `Assert` class or classes from `FluentAssertions` to assert test outcomes (just as in integration test).
- This file defines the class `FlixtubeTests` that implements the generic interface `IClassFixture<PlaywrightFixture>`.
  - Notice the type parameter `PlaywrightFixture` which is the test fixture class defined in `PlaywrightFixture.cs`.
- At the top of the file, we declare a number of private attributes, which we set in the `FlixtubeTests` constructor.
- The constructor:
  - Dependency injects `PlaywrightFixture` (an instance of our test fixture class) and `ITestOutputHelper` that we can use to output debug messages to the console when running a test.
  - Extracts the public propeties from the dependency injected `PlaywrightFixture` instance, and assigns them to the private attributes.

#### End-To-End Testing via `Flixtube.Web`'s web interface

- In the method `HomePage_NavigateTo_Title_ShouldContain_VideoList()`:
  - We create a new `context` from the Chromium-based `IBrowser` instance (i.e. we are using an instance of a Chromium-based browser to do the testing).
  - Then we create a new `page` instance from the `context`.
    - This `page` object is then used to interact with `Flixtube.Web`'s HTML elements.
  - We use the `page.GotoAsync()` method to navigate to the Home page.
  - Then we use the `page.TitleAsync()` method to read the contents for the `<title>` HTML element on the Home page, followed by a FluentAssertion to assert the title has the value `Video List`.
  - Next we use the `page.ScreenshotAsync()` method to save a screenshot of the current web page to the filesystem.
  - We then use the `page.GetByRole()` method followed by the `ClickAsync()` method to click on the `Show Viewing History` button, which will navigate to the `Viewing History` page.
  - Then we once again, use the `page.TitleAsync()` method to read the contents for the `<title>` HTML element on the Viewing History page, followed by a FluentAssertion to assert the title has the value `Viewing History`.
  - Next we, once again, use the `page.ScreenshotAsync()` method to save a screenshot of the current web page to the filesystem.
  - We then, once again, use the `page.GetByRole()` method followed by the `ClickAsync()` method to click on the `Home` button, which will navigate back to the `Home` page.
  - Lastly, we once again, use the `page.TitleAsync()` method to read the contents for the `<title>` HTML element on the Home page, followed by a FluentAssertion to assert the title has the value `Video List`.  
  - The test concludes with a call to `context.CloseAsync()` to properly dispose of the web page and Chromium-based web browser.

#### Running the End-To-End Tests with `dotnet test`

If we just wanted to end-to-end test the application via the `Web` microservice, we could:

- Create a solution file (with e.g. `dotnet new sln -n Flixtube.Metadata.sln`).
- Add `Flixtube.Web` and `Flixtube.Web.EndToEndTests` to the solution `Flixtube.Web.sln`.
- Right-click the solution file `Flixtube.Web.sln` an choose `Open Solution`.
- Start the microservice and make sure all its dependencies are avaialble (in this case, all the other microservices, two SQL Server instances, and a RabbitMQ broker instance).
- Switch to the `Testing` view in VSCode and run the tests.
- Stop the microservice and clean up its dependencies if needed (in this case, all the other microservices, two SQL Server instances, and a RabbitMQ broker instance).

A simpler way is just to use the `dotnet` CLI:

- Start the microservice and make sure all its dependencies are avaialble (in this case, all the other microservices, two SQL Server instances, and a RabbitMQ broker instance).
- Run the tests
  - Alternative 1: Move into the folder that contains the solution file `Flixtube.Web.sln` and run `dotnet test`.
  - Alternative 2: Move into the folder that contains the project file `Flixtube.Web.EndToEndTests.csproj` and run `dotnet test`.
- Stop the microservice and clean up its dependencies if needed (in this case, all the other microservices, two SQL Server instances, and a RabbitMQ broker instance).

Let's try the first alternative in the cells below.

**Note!**

- I am using **Docker Compose** to start ALL the microservices, SQLServer and RabbitMQ in Docker containers (first cell below).
- Then I am using **Docker** to run the tests in the Web microservice Docker container (second cell below).
- Finally, I am using **Docker Compose** to stop ALL the microservices, SQLServer and RabbitMQ Docker containers (third cell below).
- **We haven't gone through how to use Docker or Docker Compose yet, but we'll learn how to do this during the next workshop.**
- The `Flixtube.Web.sln` I am invoking with `dotnet test` inside the Web microservice Docker container contains two projects (`Flixtube.Web` and `Flixtube.Web.EndToEndTests`) which is why the end-to-end tests are run (second cell below).

Now, let's execute the cells below and observe the test results.

- As you can see, the final row in the output from the second cell below is:
  - `Passed!` (all tests passed)
  - `Failed: 0` (0 tests failed)
  - `Passed: 1` (1 tests passed)
  - `Skipped: 0` (no tests were skipped)
  - `Total: 1` (a total of 1 tests were found)
  - `Duration: 4 s` (the total testing time was 4 seconds)
  - `Flixtube.Web.EndToEndTests.dll (net9.0)` (the test assembly run)

In [6]:
!docker compose -f ../flixtube/compose/docker-compose-web-dev.yml --project-directory ../flixtube up -d --build --force-recreate

#0 building with "desktop-linux" instance using docker driver

#1 [metadata internal] load build definition from Dockerfile-dev
#1 transferring dockerfile: 454B 0.0s done
#1 DONE 0.8s

#2 [history internal] load build definition from Dockerfile-dev
#2 transferring dockerfile: 285B 0.1s done
#2 DONE 0.8s

#3 [minio-storage internal] load build definition from Dockerfile-dev
#3 transferring dockerfile: 300B 0.1s done
#3 DONE 0.9s

#4 [minio-storage internal] load metadata for mcr.microsoft.com/dotnet/sdk:9.0
#4 DONE 1.4s

#5 [metadata internal] load .dockerignore
#5 transferring context: 393B 0.0s done
#5 ...

#6 [minio-storage internal] load .dockerignore
#6 transferring context: 393B 0.0s done
#6 DONE 1.2s

#7 [history  1/12] FROM mcr.microsoft.com/dotnet/sdk:9.0@sha256:84fd557bebc64015e731aca1085b92c7619e49bdbe247e57392a43d92276f617
#7 DONE 0.0s

#8 [history  2/12] WORKDIR /src
#8 CACHED

#5 [metadata internal] load .dockerignore
#5 DONE 1.2s

#9 [history internal] load .dockerignore


 Service metadata  Building
 Service minio-storage  Building
 Service history  Building
 Service history  Built
 Service minio-storage  Built
 Service video-upload  Building
 Service video-streaming  Building
 Service metadata  Built
 Service video-streaming  Built
 Service video-upload  Built
 Service gateway  Building
 Service gateway  Built
 Service web  Building
 Service web  Built
 Network flixtube_default  Creating
 Network flixtube_default  Created
 Volume "flixtube_rabbit_data"  Creating
 Volume "flixtube_rabbit_data"  Created
 Volume "flixtube_sqlserver_data"  Creating
 Volume "flixtube_sqlserver_data"  Created
 Volume "flixtube_minio_data"  Creating
 Volume "flixtube_minio_data"  Created
 Container rabbit  Creating
 Container sqlserver  Creating
 Container minio  Creating
 Container sqlserver  Created
 Container minio  Created
 Container minio-mc  Creating
 Container video-storage  Creating
 Container rabbit  Created
 Container history  Creating
 Container metadata  Creating



#65 138.4 Unpacking libxkbcommon0:amd64 (1.5.0-1) ...
#65 138.5 Selecting previously unselected package libxrandr2:amd64.
#65 138.5 Preparing to unpack .../222-libxrandr2_2%3a1.5.2-2+b1_amd64.deb ...
#65 138.5 Unpacking libxrandr2:amd64 (2:1.5.2-2+b1) ...
#65 138.6 Selecting previously unselected package x11-common.
#65 138.6 Preparing to unpack .../223-x11-common_1%3a7.7+23_all.deb ...
#65 138.6 Unpacking x11-common (1:7.7+23) ...
#65 138.7 Selecting previously unselected package libxss1:amd64.
#65 138.7 Preparing to unpack .../224-libxss1_1%3a1.2.3-1_amd64.deb ...
#65 138.7 Unpacking libxss1:amd64 (1:1.2.3-1) ...
#65 138.8 Selecting previously unselected package libsdl2-2.0-0:amd64.
#65 138.8 Preparing to unpack .../225-libsdl2-2.0-0_2.26.5+dfsg-1_amd64.deb ...
#65 138.8 Unpacking libsdl2-2.0-0:amd64 (2.26.5+dfsg-1) ...
#65 138.9 Selecting previously unselected package timgm6mb-soundfont.
#65 138.9 Preparing to unpack .../226-timgm6mb-soundfont_1.3-5_all.deb ...
#65 139.0 Unpacking 

In [8]:
!docker exec web dotnet test

  Determining projects to restore...
  All projects are up-to-date for restore.
  Flixtube.Web.EndToEndTests -> /src/Flixtube.Web.EndToEndTests/bin/Debug/net9.0/Flixtube.Web.EndToEndTests.dll
Test run for /src/Flixtube.Web.EndToEndTests/bin/Debug/net9.0/Flixtube.Web.EndToEndTests.dll (.NETCoreApp,Version=v9.0)
VSTest version 17.12.0 (x64)

Starting test execution, please wait...
A total of 1 test files matched the specified pattern.

Passed!  - Failed:     0, Passed:     1, Skipped:     0, Total:     1, Duration: 4 s - Flixtube.Web.EndToEndTests.dll (net9.0)


In [9]:
!docker compose -f ../flixtube/compose/docker-compose-web-dev.yml --project-directory ../flixtube down --rmi local --volumes

 Container web  Stopping
 Container minio-mc  Stopping
 Container minio-mc  Stopped
 Container minio-mc  Removing
 Container minio-mc  Removed
 Container web  Stopped
 Container web  Removing
 Container web  Removed
 Container gateway  Stopping
 Container gateway  Stopped
 Container gateway  Removing
 Container gateway  Removed
 Container video-upload  Stopping
 Container metadata  Stopping
 Container history  Stopping
 Container video-streaming  Stopping
 Container video-streaming  Stopped
 Container video-streaming  Removing
 Container metadata  Stopped
 Container metadata  Removing
 Container video-streaming  Removed
 Container history  Stopped
 Container history  Removing
 Container metadata  Removed
 Container video-upload  Stopped
 Container video-upload  Removing
 Container history  Removed
 Container sqlserver  Stopping
 Container video-upload  Removed
 Container video-storage  Stopping
 Container rabbit  Stopping
 Container rabbit  Stopped
 Container rabbit  Removing
 Containe

---

### End-To-End Test additional scenarios via the `Flixtube.Web` Microservice

- As an exercise, try:
  - Adding additional tests to the `FlixtubeTests.cs` file in the `Flixtube.Wen.EndToEndTests` project.
  - Run the same commands as in the three notebook cells above.

---

### Conclusion

To learn more about Playwright, visit the [Playwright Introduction](https://playwright.dev/dotnet/docs/intro).

This completes the introduction to end-to-end testing with xUnit, FluentAssertions and Playwright in VSCode, where we have end-to-end tested the Flixtube application via it's web-based frontend. This also concludes the workshop.